In [ ]:
import base64
import subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
token = secrets.get_secret("GITHUB_TOKEN")

username = "AnikMallick"
repo_name = "bn-proverbs-in-context"

repo_dir = Path("/kaggle/working") / repo_name

auth = base64.b64encode(
    f"{username}:{token}".encode()
).decode()

try:
    subprocess.run(
        [
            "git",
            "-c",
            f"http.https://github.com/.extraheader="
            f"AUTHORIZATION: basic {auth}",
            "clone",
            f"https://github.com/{username}/{repo_name}.git",
            str(repo_dir),
        ],
        check=True,
    )
finally:
    del token
    del auth

print("Repository cloned successfully.")

In [ ]:
%cd /kaggle/working/bn-proverbs-in-context

!pip install -e .

In [1]:
from pathlib import Path
from probad.schema import TestItem, validate_items
import json
from typing import Any

DATA_PATH = Path("/kaggle/input/datasets/amallick0507/probad")

train_data = list(TestItem.load_items(DATA_PATH/"train.jsonl"))
test_data = list(TestItem.load_items(DATA_PATH/"test.jsonl"))
val_data = list(TestItem.load_items(DATA_PATH/"val.jsonl"))


source = DATA_PATH / "stopwords-bn.json"

try:
    data: Any = json.loads(source.read_text(encoding="utf-8-sig"))
except FileNotFoundError as exc:
    raise FileNotFoundError(f"Stopword file not found: {source}") from exc
except json.JSONDecodeError as exc:
    raise ValueError(
        f"Invalid JSON in stopword file {source}: "
        f"line {exc.lineno}, column {exc.colno}: {exc.msg}"
    ) from exc

BN_STOPWORDS =  frozenset([word.strip() for word in data])

In [2]:
from probad.utils import print_div

print_div("train")
for issue in validate_items(train_data, BN_STOPWORDS):
    print(issue)

print_div("test")
for issue in validate_items(test_data, BN_STOPWORDS):
    print(issue)

print_div("val")
for issue in validate_items(val_data, BN_STOPWORDS):
    print(issue)

=============================================> train <==============================================
==============================================> test <==============================================
[WARNING] prv_088_1: en.correct_echoes_gloss: correct option reuses many literal-gloss words
[WARNING] prv_088_2: en.correct_echoes_gloss: correct option reuses many literal-gloss words
==============================================> val <===============================================


In [3]:
import probad.builder as exp

In [11]:
print_div("e1 maps")
print("|| memorization ||")
r = exp.e1_memorisation(train_data[0].model_dump(), render="bengali", template=0)
print(r[0])
print(">expected ans")
print(r[1])
print("====")
r = exp.e1_memorisation(train_data[0].model_dump(), render="english", template=1)
print(r[0])
print(">expected ans")
print(r[1])


print_div("e2 maps")
print("|| reasoning ||")
r = exp.e2_reasoning(train_data[0].model_dump())
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])
print("==== my test, literal distractor")
r = exp.e2_reasoning(train_data[0].model_dump(), keys=("correct", "literal"))
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])

print_div("e5 maps")
print("|| reasoning cotrol ||")
r = exp.e5_control(train_data[0].model_dump())
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])
print("==== negetion")
r = exp.e5_negated(train_data[0].model_dump(), question="neg") #"neg", "neg1", "neg2", "neg3", "neg4"
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])

print_div("e6 maps")
print("|| translation ||")
r = exp.e6_translation(train_data[0].model_dump(), condition="bn_native")#"bn_native", "en_romanised", "en_gloss"
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])
print("====")
r = exp.e6_translation(train_data[0].model_dump(), condition="en_romanised")#"bn_native", "en_romanised", "en_gloss"
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])
print("====")
r = exp.e6_translation(train_data[0].model_dump(), condition="en_gloss")#"bn_native", "en_romanised", "en_gloss"
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])

print_div("e7 maps")
print("|| few shot ||")
r = exp.e7_few_shot(train_data[0].model_dump(),demo_items=[v.model_dump() for v in val_data], k=2)
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])

print_div("e9 maps")
print("|| template ||")
r = exp.e9_template(train_data[0].model_dump(),template=1)
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])
print("====")
r = exp.e9_template(train_data[0].model_dump(),template=2)
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])
print("====")
r = exp.e9_template(train_data[0].model_dump(),template=3)
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])


============================================> e1 maps <=============================================
|| memorization ||
প্রবাদ: ইটটি মারলে পাটকেলটি খেতে
>expected ans
হয়
====
complete this proverb: itti marle patkelti khete
>expected ans
hoy
============================================> e2 maps <=============================================
|| reasoning ||
question: what does the person mean by the proverb?
proverb:'ইটটি মারলে পাটকেলটি খেতে হয়'
context: রাজেশ: সুমন সপ্তাহের পর সপ্তাহ অনলাইনে প্রাক্তন সহকর্মীকে ট্রোল করেছিল, আর এখন সে সুমনের পুরোনো লজ্জার সব মেসেজ পোস্ট করে দিয়েছে। ইপ্সিতা: ইটটি মারলে পাটকেলটি খেতে হয়।
choices: a: ইপ্সিতার ধারণা ঝগড়াটা আসলে প্রাক্তন সহকর্মীই প্রথমে শুরু করেছিল। b: সুমন আগে আঘাত করেছিল, তাই পাল্টা আঘাত আসাটাই স্বাভাবিক ছিল।
answer:
>expected ans
b
['a', 'b']
==== my test, literal distractor
question: what does the person mean by the proverb?
proverb:'ইটটি মারলে পাটকেলটি খেতে হয়'
context: রাজেশ: সুমন সপ্তাহের পর সপ্তাহ অনলাইনে প্রাক্তন সহকর্মীকে ট্র

In [12]:
print_div("x1 mine")
print("|| x1_correct_vs_literal ||")
r = exp.x1_correct_vs_literal(train_data[0].model_dump(),lang="en")
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])
print("====")
r = exp.x1_correct_vs_literal(train_data[0].model_dump(),lang="bn")
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])


print_div("x2 mine")
print("|| negative ||")
r = exp.x2_negative(train_data[0].model_dump(),lang="en")
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])
print("====")
r = exp.x2_negative(train_data[0].model_dump(),lang="bn")
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])


print_div("x3 mine")
print("|| none ||")
r = exp.x3_none_of_these(train_data[0].model_dump(),lang="en")
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])
print("====")
r = exp.x3_none_of_these(train_data[0].model_dump(),lang="bn")
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])


print_div("x4 mine")
print("|| meaning_no_context ||")
r = exp.x4_meaning_no_context(train_data[0].model_dump(),lang="en")
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])
print("====")
r = exp.x4_meaning_no_context(train_data[0].model_dump(),lang="bn")
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])


print_div("x5 mine")
print("|| three_way ||")
r = exp.x5_three_way(train_data[0].model_dump(),lang="en")
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])
print("====")
r = exp.x5_three_way(train_data[0].model_dump(),lang="bn")
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])

print_div("x6 mine")
print("|| render ||")
r = exp.x6_render(train_data[0].model_dump(),render="romanised")
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])
print("====")
r = exp.x6_render(train_data[0].model_dump(),render="gloss")
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])
print("====")
r = exp.x6_render(train_data[0].model_dump(),render="gloss_only")
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])
print("====")
r = exp.x6_render(train_data[0].model_dump(),render="bengali")
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])


print_div("x7 mine")
print("|| no_question ||")
r = exp.x7_no_question(train_data[0].model_dump(), lang="bn")
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])
print("====")
r = exp.x7_no_question(train_data[0].model_dump(), lang="en")
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])

print_div("x8 mine")
print("|| with prompt ||")
r = exp.x8_system_prompt(train_data[0].model_dump(), condition="C0")
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])
print("====")
r = exp.x8_system_prompt(train_data[0].model_dump(), condition="C1a")
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])
print("====")
r = exp.x8_system_prompt(train_data[0].model_dump(), condition="C1b")
print(r[0])
print(">expected ans")
print(r[1])
print(r[2])


============================================> x1 mine <=============================================
|| x1_correct_vs_literal ||
Rajesh: Sumon trolled his ex-colleague online for weeks, and now she has posted all his old embarrassing messages.
Ipsita: Itti marle patkelti khete hoy.

Question: What does Ipsita mean?
A. Throw a brick at someone and a stone comes back at you.
B. Sumon hurt her first, so he should have expected to be hurt back.
Answer:
>expected ans
b
['a', 'b']
====
রাজেশ: সুমন সপ্তাহের পর সপ্তাহ অনলাইনে প্রাক্তন সহকর্মীকে ট্রোল করেছিল, আর এখন সে সুমনের পুরোনো লজ্জার সব মেসেজ পোস্ট করে দিয়েছে।
ইপ্সিতা: ইটটি মারলে পাটকেলটি খেতে হয়।

প্রশ্ন: ইপ্সিতা কী বোঝাতে চাইছে?
A. কাউকে লক্ষ্য করে ইট ছুড়লে উল্টো দিক থেকে একটা ঢিল এসে নিজের গায়েই লাগে।
B. সুমন আগে আঘাত করেছিল, তাই পাল্টা আঘাত আসাটাই স্বাভাবিক ছিল।
উত্তর:
>expected ans
b
['a', 'b']
============================================> x2 mine <=============================================
|| negative ||
Rajesh: Sumon trolled

In [ ]:
# from kaggle_secrets import UserSecretsClient
# import os
# user_secrets = UserSecretsClient()
# secret_value_1 = user_secrets.get_secret("HF_TOKEN")
# os.environ['HF_TOKEN'] = secret_value_1

# MODEL_NAME = "allenai/OLMo-2-0425-1B"
# from probad.scoring import load_model

# model, tok = load_model(MODEL_NAME)

In [8]:
from probad.scoring import answer_one, generate

In [13]:

# r = exp.x6_render(train_data[0].model_dump(),render="romanised")
# r = exp.x3_none_of_these(train_data[0].model_dump(),lang="bn")
r = exp.x1_correct_vs_literal(train_data[0].model_dump(),lang="bn")
print(answer_one(model, tok, r[0], r[1], r[2], system=exp.SYSTEM_PROMPTS["C1b"], chat=False))
r[0], r[1]

{'gold': 'b', 'pred': 'a', 'correct': False, 'p_a': 0.920217986535907, 'p_b': 0.079782013464093, 'margin': -2.4453125, 'letter_mass': 4.1988092456920665e-05}


('রাজেশ: সুমন সপ্তাহের পর সপ্তাহ অনলাইনে প্রাক্তন সহকর্মীকে ট্রোল করেছিল, আর এখন সে সুমনের পুরোনো লজ্জার সব মেসেজ পোস্ট করে দিয়েছে।\nইপ্সিতা: ইটটি মারলে পাটকেলটি খেতে হয়।\n\nপ্রশ্ন: ইপ্সিতা কী বোঝাতে চাইছে?\nA. কাউকে লক্ষ্য করে ইট ছুড়লে উল্টো দিক থেকে একটা ঢিল এসে নিজের গায়েই লাগে।\nB. সুমন আগে আঘাত করেছিল, তাই পাল্টা আঘাত আসাটাই স্বাভাবিক ছিল।\nউত্তর:',
 'b')

In [14]:

r = exp.e1_memorisation(train_data[0].model_dump(), render="bengali", template=0)
print(generate(model, tok, r[0], max_new_tokens=6))
r[0], r[1]

 পারেন


('প্রবাদ: ইটটি মারলে পাটকেলটি খেতে', 'হয়')

In [15]:
print_div("e1 maps")
print("|| memorization ||")
r = exp.e1_memorisation(train_data[0].model_dump(), render="bengali", template=0)
print(generate(model, tok, r[0], max_new_tokens=6))
print("====")
r = exp.e1_memorisation(train_data[0].model_dump(), render="english", template=1)
print(generate(model, tok, r[0], max_new_tokens=6))

print_div("e2 maps")
print("|| reasoning ||")
r = exp.e2_reasoning(train_data[0].model_dump())
print(answer_one(model, tok, r[0], r[1],r[2], system=None, chat=False))
print("==== my test, literal distractor")
r = exp.e2_reasoning(train_data[0].model_dump(), keys=("correct", "literal"))
print(answer_one(model, tok, r[0], r[1],r[2], system=None, chat=False))

print_div("e5 maps")
print("|| reasoning cotrol ||")
r = exp.e5_control(train_data[0].model_dump())
print(answer_one(model, tok, r[0], r[1], r[2],system=None, chat=False))
print("==== negetion")
r = exp.e5_negated(train_data[0].model_dump(), question="neg") #"neg", "neg1", "neg2", "neg3", "neg4"
print(answer_one(model, tok, r[0], r[1],r[2], system=None, chat=False))

print_div("e6 maps")
print("|| translation ||")
r = exp.e6_translation(train_data[0].model_dump(), condition="bn_native")#"bn_native", "en_romanised", "en_gloss"
print(answer_one(model, tok, r[0], r[1], r[2],system=None, chat=False))
print("====")
r = exp.e6_translation(train_data[0].model_dump(), condition="en_romanised")#"bn_native", "en_romanised", "en_gloss"
print(answer_one(model, tok, r[0], r[1], r[2],system=None, chat=False))
print("====")
r = exp.e6_translation(train_data[0].model_dump(), condition="en_gloss")#"bn_native", "en_romanised", "en_gloss"
print(answer_one(model, tok, r[0], r[1], r[2],system=None, chat=False))

print_div("e7 maps")
print("|| few shot ||")
r = exp.e7_few_shot(train_data[0].model_dump(),demo_items=[v.model_dump() for v in val_data], k=2)
print(answer_one(model, tok, r[0], r[1], r[2],system=None, chat=False))


print_div("e9 maps")
print("|| template ||")
r = exp.e9_template(train_data[0].model_dump(),template=1)
print(answer_one(model, tok, r[0], r[1], r[2],system=None, chat=False))
print("====")
r = exp.e9_template(train_data[0].model_dump(),template=2)
print(answer_one(model, tok, r[0], r[1], r[2],system=None, chat=False))
print("====")
r = exp.e9_template(train_data[0].model_dump(),template=3)
print(answer_one(model, tok, r[0], r[1], r[2],system=None, chat=False))

============================================> e1 maps <=============================================
|| memorization ||
 পারেন
====
, khaile khete
============================================> e2 maps <=============================================
|| reasoning ||
{'gold': 'b', 'pred': 'a', 'correct': False, 'p_a': 0.81227240058765, 'p_b': 0.18772759941234998, 'margin': -1.4648437798023224, 'letter_mass': 0.8724912545326939}
==== my test, literal distractor
{'gold': 'b', 'pred': 'a', 'correct': False, 'p_a': 0.8322366527818008, 'p_b': 0.16776334721819924, 'margin': -1.6015625, 'letter_mass': 0.8444451433765134}
============================================> e5 maps <=============================================
|| reasoning cotrol ||
{'gold': 'b', 'pred': 'a', 'correct': False, 'p_a': 0.86613252261295, 'p_b': 0.13386747738705002, 'margin': -1.8671875894069672, 'letter_mass': 0.9340244645825007}
==== negetion
{'gold': 'a', 'pred': 'a', 'correct': True, 'p_a': 0.8158201664107081, 'p_b': 0.

In [16]:
r = exp.e5_negated(train_data[1].model_dump(), question="neg")
print(r)
print(answer_one(model, tok, r[0], r[1], r[2],system=None, chat=False))

("question: what does the person not mean by the proverb?\nproverb:'ইটটি মারলে পাটকেলটি খেতে হয়'\ncontext: রাসেল: ভাড়াটেদের তাড়াতে বাড়িওয়ালা পানি বন্ধ করে দিয়েছিল, আর এখন ওরা রাজউকে ওর অবৈধ তলাগুলোর কথা জানিয়ে দিয়েছে। হালিমা: ইটটি মারলে পাটকেলটি খেতে হয়।\nchoices: a: হালিমার ধারণা ভাড়া কমানোর জন্যই ভাড়াটেরা বাড়িওয়ালার নামে নালিশ করেছে। b: ক্ষতি করেছিল বলে বাড়িওয়ালাকে এখন পাল্টা ক্ষতি সইতে হচ্ছে।\nanswer:", 'a', ['a', 'b'])
{'gold': 'a', 'pred': 'a', 'correct': True, 'p_a': 0.8572908574590595, 'p_b': 0.1427091425409405, 'margin': 1.7929686605930328, 'letter_mass': 0.8734824944199086}


In [5]:
torch.cuda.is_available()

True

In [4]:

import torch
import queue
import threading
from transformers import AutoTokenizer, AutoModelForCausalLM


class LoadBalancedModel:

    def __init__(self, name, gpu_ids=(0, 1), dtype=None):

        self.models = []
        self.devices = []
        self.pool = queue.Queue()
        self.local = threading.local()

        if dtype is None:
            dtype = torch.float16

        self.tok = AutoTokenizer.from_pretrained(name)

        for gpu_id in gpu_ids:
            device = f"cuda:{gpu_id}"

            model = AutoModelForCausalLM.from_pretrained(
                name,
                torch_dtype=dtype
            ).to(device).eval()

            self.models.append(model)
            self.devices.append(device)
            self.pool.put(len(self.models) - 1)

    @property
    def device(self):
        # Reserve a free model for this thread
        idx = self.pool.get()
        self.local.idx = idx
        return self.devices[idx]

    def _release(self):
        idx = getattr(self.local, "idx", None)

        if idx is not None:
            del self.local.idx
            self.pool.put(idx)

    def __call__(self, *args, **kwargs):
        idx = getattr(self.local, "idx", None)

        if idx is None:
            idx = self.pool.get()

        try:
            return self.models[idx](*args, **kwargs)
        finally:
            self.local.idx = None
            self.pool.put(idx)

    def generate(self, *args, **kwargs):
        idx = getattr(self.local, "idx", None)

        if idx is None:
            idx = self.pool.get()

        try:
            return self.models[idx].generate(*args, **kwargs)
        finally:
            self.local.idx = None
            self.pool.put(idx)


In [6]:
from kaggle_secrets import UserSecretsClient
import os
user_secrets = UserSecretsClient()
secret_value_1 = user_secrets.get_secret("HF_TOKEN")
os.environ['HF_TOKEN'] = secret_value_1

MODEL_NAME = "allenai/OLMo-2-0425-1B"
model = LoadBalancedModel(
    MODEL_NAME,
    gpu_ids=(0, 1)
)

tok = model.tok

config.json:   0%|          | 0.00/623 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.34k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.14M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/14.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

In [9]:
r = exp.e5_negated(train_data[1].model_dump(), question="neg")
print(r)
print(answer_one(model, tok, r[0], r[1], r[2],system=None, chat=False))

("question: what does the person not mean by the proverb?\nproverb:'ইটটি মারলে পাটকেলটি খেতে হয়'\ncontext: রাসেল: ভাড়াটেদের তাড়াতে বাড়িওয়ালা পানি বন্ধ করে দিয়েছিল, আর এখন ওরা রাজউকে ওর অবৈধ তলাগুলোর কথা জানিয়ে দিয়েছে। হালিমা: ইটটি মারলে পাটকেলটি খেতে হয়।\nchoices: a: হালিমার ধারণা ভাড়া কমানোর জন্যই ভাড়াটেরা বাড়িওয়ালার নামে নালিশ করেছে। b: ক্ষতি করেছিল বলে বাড়িওয়ালাকে এখন পাল্টা ক্ষতি সইতে হচ্ছে।\nanswer:", 'a', ['a', 'b'])
{'gold': 'a', 'pred': 'a', 'correct': True, 'p_a': 0.8572908574590595, 'p_b': 0.1427091425409405, 'margin': 1.7929686605930328, 'letter_mass': 0.8734824944199086}


In [10]:
from concurrent.futures import ThreadPoolExecutor

def process_item(item):
    r = exp.e5_negated(item.model_dump(), question="neg")

    return answer_one(
        model, tok,
        r[0], r[1], r[2],
        system=None,
        chat=False
    )


with ThreadPoolExecutor(max_workers=2) as executor:
    results = list(executor.map(process_item, train_data[:10]))

print(results)

[{'gold': 'a', 'pred': 'a', 'correct': True, 'p_a': 0.8158201664107081, 'p_b': 0.18417983358929188, 'margin': 1.4882813096046448, 'letter_mass': 0.8854844962081962}, {'gold': 'a', 'pred': 'a', 'correct': True, 'p_a': 0.8572908574590595, 'p_b': 0.1427091425409405, 'margin': 1.7929686605930328, 'letter_mass': 0.8734824944199086}, {'gold': 'b', 'pred': 'a', 'correct': False, 'p_a': 0.8250265356639277, 'p_b': 0.17497346433607228, 'margin': -1.5507812201976776, 'letter_mass': 0.8255447046668337}, {'gold': 'b', 'pred': 'a', 'correct': False, 'p_a': 0.7846492519854912, 'p_b': 0.21535074801450882, 'margin': -1.2929687201976776, 'letter_mass': 0.8737840566984001}, {'gold': 'a', 'pred': 'a', 'correct': True, 'p_a': 0.8344068904870238, 'p_b': 0.1655931095129762, 'margin': 1.6171875298023224, 'letter_mass': 0.9167319967463197}, {'gold': 'a', 'pred': 'a', 'correct': True, 'p_a': 0.8529354795409523, 'p_b': 0.14706452045904775, 'margin': 1.7578125, 'letter_mass': 0.9232910070886903}, {'gold': 'b', 'p